In [ ]:
from __future__ import annotations

import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
from logdir import LogDir

from gcngrasp import Path
from gcngrasp.system.api import get_da3_mapper
from gcngrasp.system.data_type.workspace import IrregularWorkspace
from gcngrasp.system.system import System, Scene

warnings.filterwarnings("ignore")

WORKSPACE = IrregularWorkspace("config/system/franka_ws.npy", r_decay=.9)
MAPPER = get_da3_mapper()

# Robot

In [ ]:
import my_robot
from gcngrasp.api.utils import pose_tf
from gcngrasp.api.utils.camera import Frame
from gcngrasp.system import process_config

CONFIG = process_config("config/system/params.yaml")

# config
addr_robot = "192.168.99.186:5555"
z_ground = .1091
focus = np.array([.4, 0., .35])

# initial
robot_ctrl = my_robot.get_robot_client(addr=addr_robot)
robot_ctrl = my_robot.CuRoboWrapper(robot_ctrl, z_ground=z_ground)
print(robot_ctrl)
robot_ctrl.grip_reset()
robot_ctrl.go_home()


def move_and_observ(pos: np.ndarray,
                    plt_show: bool = False,
                    save_to: Path = None) -> Frame:
    Tcb = pose_tf.lookat_to_Tcw(pos, focus)
    assert robot_ctrl.move(Tcb @ pose_tf.TFmanager["hand", "camera"])
    frame = robot_ctrl.get_frame()
    # visualize
    if plt_show: plt.imshow(frame.concat())
    if save_to:
        save_to.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(save_to), frame.concat()[..., ::-1])
    return frame


idx, lookat = 2, np.array([0, 1, -.3])     # pot
# idx, lookat = 1, np.array([0, 1, -1.2])     # brush
# idx, lookat = 4, np.array([.5, 0, -1])     # cup
lookat /= np.linalg.norm(lookat)
move_and_observ(pos=focus - lookat * CONFIG["planner"]["dist_obs"], plt_show=True)

# Main

In [ ]:
ASSETS = sorted(Path("assets").iterdir())
folder = ASSETS[idx]

max_nbv = 1
args = folder.name.split("--")[1:]

scene = Scene(CONFIG, target_obj=args[0], tog_model_args=args[1:3], workspace=WORKSPACE)
print(scene)

system = System(CONFIG)
logdir = LogDir(scene.target_obj, rootdir="runs/nbv_online")

# update
for i in range(max_nbv + 1):
    frame = robot_ctrl.get_frame()
    plt.imshow(frame.concat())

    scene.integrate(frame, mapper=MAPPER if i else None)
    act = system.get_action(scene, plan_nbv=i != max_nbv, thresh_grasp=.9)
    print(act)

    # visualize
    scene.save_info(logdir.pdir(f"iter{i}"), next_Tcw=act[0].Tcw)
    o3d.visualization.draw_geometries(scene.to_o3d(next_Tcw=act[0].Tcw))
    if input("Interrupt (y/*) ?: ").strip().lower() == "y": break

    # execute
    try:
        if act.grasp:
            robot_ctrl.grasp(act.Teb, is_lift=True, d_approach=.1, d_retract=.1)
        else:
            robot_ctrl.move(act[0].Teb)

    except:
        for j, Teb in enumerate(act[0].to_Tebs(d_pre=.1, d_lift=.1)):
            robot_ctrl.move(Teb)
            # input("Press enter to continue...")
            if j == 1: robot_ctrl.grip(True)
    if act.grasp: break